# Exoplanet Transit — Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TAITeachingWithAI/ScientificAILab/blob/main/notebooks/exoplanet_transit_analysis.ipynb)

This notebook helps you load and analyse the light-sensor data from your model-exoplanet experiment.

It works with a CSV exported from **Physics Toolbox**, the **Arduino Science Journal**, or **any plain CSV** with a time column and a light/brightness column.

**You will produce:**
1. a **light curve** (light vs. time),
2. a **transit depth** value,
3. a plot of **transit depth vs. planet size²** across the planet sizes you tested.

Run the cells from top to bottom. Where a cell says *you can change this*, edit the marked line if the automatic guess is wrong.

## 1. Upload your data

First, in your phone app, **export your recording as a CSV** and send it to your computer:
- **Physics Toolbox:** stop the recording, tap the share/export button, choose CSV.
- **Arduino Science Journal:** open the experiment, export it, and choose CSV.

Then run the cell below and choose that CSV file.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()          # opens a file picker
    filename = next(iter(uploaded))
except Exception:
    # Not on Colab: put your CSV next to the notebook and set its name here.
    uploaded = {}
    filename = 'your_data.csv'
print('Using file:', filename)

## 2. Load and preview the data

This reads the CSV and tries to find the **time** column and the **light** column automatically. If it guesses wrong, set `time_col` / `light_col` yourself in the next cell.

In [ ]:
import io
import numpy as np
import pandas as pd

def load_csv(name):
    if name in uploaded:
        raw = uploaded[name].decode('utf-8', errors='replace')
    else:
        with open(name, 'r', encoding='utf-8', errors='replace') as f:
            raw = f.read()
    # Try common separators and keep the one that yields >= 2 columns.
    for sep in [',', ';', '\t']:
        try:
            d = pd.read_csv(io.StringIO(raw), sep=sep, engine='python')
            if d.shape[1] >= 2:
                return d
        except Exception:
            pass
    return pd.read_csv(io.StringIO(raw))

df = load_csv(filename)
df.columns = [str(c).strip() for c in df.columns]
print('Columns found:', list(df.columns))
df.head()

In [ ]:
def pick_column(df, keywords):
    for c in df.columns:
        if any(k in c.lower() for k in keywords):
            return c
    return None

time_col  = pick_column(df, ['time', 'timestamp', 'seconds', 'elapsed'])
light_col = pick_column(df, ['illuminance', 'lux', 'lx', 'light', 'bright', 'intensity', 'value'])

# Fallbacks for unusual headers (e.g. a plain CSV):
if time_col is None:
    time_col = df.columns[0]                       # assume the first column is time
if light_col is None:
    for c in df.columns:                           # first numeric column that isn't time
        if c != time_col and pd.to_numeric(df[c], errors='coerce').notna().mean() > 0.5:
            light_col = c
            break

# --- you can change this: if a guess is wrong, set the names by hand, e.g. ---
# time_col  = 'time'
# light_col = 'Illuminance (lx)'

print('Time column :', time_col)
print('Light column:', light_col)
assert time_col and light_col, 'Could not detect the columns — set them by hand above.'

In [ ]:
# Turn the columns into clean numeric arrays (time in seconds from the start).
t = pd.to_numeric(df[time_col], errors='coerce')
if t.isna().mean() > 0.5:                 # looks like a date/time stamp instead of seconds
    tt = pd.to_datetime(df[time_col], errors='coerce', format='mixed')
    t = (tt - tt.min()).dt.total_seconds()
light = pd.to_numeric(df[light_col], errors='coerce')

ok = t.notna() & light.notna()
time_s = (t[ok] - t[ok].min()).to_numpy()
light  = light[ok].to_numpy()
print(f'{len(time_s)} data points over {time_s.max():.1f} s')

## 3. Plot the light curve  *(deliverable 1)*

This is your transit light curve — brightness versus time. You should see a **dip** where the model planet passed in front of the star.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(time_s, light, lw=1)
plt.xlabel('Time (s)')
plt.ylabel(f'Light  ({light_col})')
plt.title('Transit light curve')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('light_curve.png', dpi=150)
plt.show()

## 4. Measure the transit depth

The **transit depth** is how much the light dropped during the transit:

$$\text{depth}\,(\%) = \frac{\text{baseline} - \text{minimum}}{\text{baseline}} \times 100$$

- **baseline** = the normal brightness outside the dip,
- **minimum** = the lowest brightness during the dip.

The cell below does this automatically. If your recording is noisy or has more than one dip, use **Option B** to tell it exactly which time window the transit is in.

In [ ]:
# --- Option A: automatic ---
baseline = float(np.median(light))
minimum  = float(np.min(light))
depth_pct = (baseline - minimum) / baseline * 100
print(f'Automatic:  baseline = {baseline:.2f},  minimum = {minimum:.2f},  depth = {depth_pct:.2f}%')

# --- Option B: choose the transit window by hand (uncomment and set the two times) ---
# t_start, t_end = 5.0, 9.0                     # seconds: the dip region
# in_transit = (time_s >= t_start) & (time_s <= t_end)
# baseline = float(np.median(light[~in_transit]))
# minimum  = float(np.min(light[in_transit]))
# depth_pct = (baseline - minimum) / baseline * 100
# print(f'Manual:  baseline = {baseline:.2f},  minimum = {minimum:.2f},  depth = {depth_pct:.2f}%')

# Show the baseline and minimum on the light curve.
plt.figure(figsize=(8, 4))
plt.plot(time_s, light, lw=1)
plt.axhline(baseline, color='green', ls='--', label='baseline')
plt.axhline(minimum, color='red', ls='--', label='minimum')
plt.xlabel('Time (s)'); plt.ylabel(f'Light ({light_col})')
plt.title(f'Transit depth = {depth_pct:.2f}%')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5. Compare planet sizes  *(deliverable 2)*

Run steps 1–4 again for **each planet size** you tested and note the transit depth each time. Then type your `(size, depth)` pairs into the list below and run the cell. It plots **transit depth vs. planet size²** and fits a straight line.

*Use a consistent measure of size (for example the planet's radius or diameter in cm).*

In [ ]:
# Fill in YOUR results: one (size, transit depth %) pair per planet you tested.
results = [
    # (size, depth_percent),
    # (1.0,  4.2),
    # (1.5,  9.1),
    # (2.0, 16.5),
]

if len(results) >= 2:
    size  = np.array([r[0] for r in results], float)
    depth = np.array([r[1] for r in results], float)
    x = size ** 2
    slope, intercept = np.polyfit(x, depth, 1)
    xx = np.linspace(0, x.max() * 1.05, 50)

    plt.figure(figsize=(7, 4))
    plt.scatter(x, depth, zorder=3)
    plt.plot(xx, slope * xx + intercept, '--', color='gray',
             label=f'fit: depth = {slope:.2f}·size² + {intercept:.2f}')
    plt.xlabel('Planet size squared'); plt.ylabel('Transit depth (%)')
    plt.title('Transit depth vs. planet size²')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig('depth_vs_size2.png', dpi=150)
    plt.show()
    print(f'Slope = {slope:.3f}. A straight line through the origin means depth is proportional to size².')
else:
    print('Add at least two (size, depth) results above, then run this cell again.')

## 6. Download your figures

This downloads the two figures so you can add them to your report.

In [ ]:
try:
    from google.colab import files
    for f in ['light_curve.png', 'depth_vs_size2.png']:
        try:
            files.download(f)
        except Exception as e:
            print('Could not download', f, '-', e)
except Exception:
    print('Not on Colab — the figures are saved next to this notebook.')

---
**Need to change the code?** Ask **Emmy** (or your own AI tutor): describe what you want to do and paste any error messages. Always check that the output uses *your* data and that the axes and units are labelled correctly — the AI can be wrong.